In [ ]:
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
import numpy as np

In [ ]:
if os.path.basename(os.getcwd()) == "src":
    os.chdir("..")

from src.data_loader import load_data

data = load_data("data")

### Dataset Roles 

**main_df (interaction data)**  

- Construct temporal learning sequences** (step-indexed per student)  

- Derive window-level features (success rate, difficulty, engagement proxy)  

- Generate system-level time series X(t) for PCMCI  

→ used to model *temporal learning dynamics and state evolution*



**experimental_df (causal data)**  

- Provide causal priors / constraints between constructs (from A/B tests)  

- Validate and filter discovered edges in the PCMCI graph  

- Support interpretation of causal directions (treatment → outcome)  

→ used to ground the learned graph in *real interventional evidence*

`main_df` → **temporal representation + structure learning input (X(t))**  
`experimental_df` → **causal validation & prior knowledge for DAG refinement** 

# 1. Main DF

In [ ]:
# Main DF
training_data = data["training_data"]
student_meta = data["student_meta"]
topic_pathway = data["topic_pathway"]
construct_prerequisites_test = data["construct_prerequisites_test"]
constructs_input_test = data["constructs_input_test"]
subject_meta = data["subject_meta"]

main_df = training_data.copy()

main_df = main_df.merge(student_meta, how='left', on='UserId')

main_df = main_df.merge(
    topic_pathway,
    how='left',
    on='QuizId',
    suffixes=('', '_topic')
)

main_df = main_df.merge(
    construct_prerequisites_test[['ConstructId', 'PrerequisiteConstructIds']],
    how='left',
    on='ConstructId'
)

main_df = main_df.merge(
    constructs_input_test.assign(is_input_test=1),
    how='left',
    on='ConstructId'
)

main_df['is_input_test'] = main_df['is_input_test'].fillna(0)

main_df = main_df.merge(
    subject_meta[['SubjectId', 'ParentId', 'Level']],
    how='left',
    on='SubjectId',
    suffixes=('', '_subject')
)

In [ ]:
#Data cleaning
df = main_df.copy()

df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df = df[df['Timestamp'].notna()]

df = df.dropna(subset=['UserId', 'ConstructId'])

df = df.sort_values(['UserId', 'Timestamp'], kind='mergesort')

df['IsCorrect'] = df['IsCorrect'].fillna(-1)
clean_main_df = df.reset_index(drop=True)

In [ ]:
interaction_counts = df.groupby('UserId').size().reset_index(name='total_interactions')

min_interactions = 20
valid_users = interaction_counts[interaction_counts['total_interactions'] >= min_interactions]['UserId']

filtered_df = df[df['UserId'].isin(valid_users)].copy()

clean_main_df = filtered_df.reset_index(drop=True)

In [ ]:
#timestamp → step index :
df = df.sort_values(['UserId', 'Timestamp']).reset_index(drop=True)

df['step_index'] = df.groupby('UserId').cumcount() + 1

clean_main_df = df.reset_index(drop=True)

In [ ]:
#Encoding & standardization
cat_cols = ['ConstructId', 'QuizId', 'SubjectId', 'ParentId', 'Level_subject']
for col in cat_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col + '_encoded'] = le.fit_transform(df[col].fillna('missing').astype(str))

df = pd.get_dummies(df, columns=['is_input_test'], prefix='is_input', dtype='int8')

num_cols = ['step_index']  
scaler = StandardScaler()

for col in num_cols:
    if col in df.columns:
        df[col + '_scaled'] = scaler.fit_transform(df[[col]])

clean_main_df = df.reset_index(drop=True)

In [ ]:
#engagement proxy definition
df = df.sort_values(['UserId', 'step_index'])
df['time_delta'] = df.groupby('UserId')['Timestamp'].diff().dt.total_seconds()

df['engagement_proxy'] = 0.0

df['is_correct'] = df['IsCorrect'].fillna(0).astype(int)
df['time_delta'] = df['time_delta'].fillna(df['time_delta'].median())

df['engagement_proxy'] = (
    (1 / (1 + df['time_delta'])) * 
    (1 + df['is_correct']) * 
    0.5
)

df['consecutive_correct'] = df.groupby('UserId')['is_correct'].rolling(window=5, min_periods=1).sum().values
df['attempts_on_same_question'] = df.groupby(['UserId', 'QuestionId']).cumcount() + 1

scaler = StandardScaler()
df['engagement_proxy_scaled'] = scaler.fit_transform(df[['engagement_proxy']])
clean_main_df = df.reset_index(drop=True)

In [ ]:
#Student cluster
student_features = df.groupby('UserId').agg(
    overall_accuracy=('IsCorrect', lambda x: (x == 1).mean()),
    total_interactions=('UserId', 'size'),
    unique_constructs=('ConstructId', 'nunique'),
    unique_subjects=('SubjectId', 'nunique'),
    accuracy_slope=('IsCorrect', lambda x: np.polyfit(range(len(x)), (x==1).astype(int), 1)[0] if len(x) > 1 else 0)
).reset_index()

if 'engagement_proxy' in df.columns:
    engagement_agg = df.groupby('UserId')['engagement_proxy'].mean().reset_index(name='engagement_mean')
    student_features = student_features.merge(engagement_agg, on='UserId', how='left')
else:
    student_features['engagement_mean'] = 0

if 'consecutive_correct' in df.columns:
    consec_agg = df.groupby('UserId')['consecutive_correct'].max().reset_index(name='consecutive_correct_max')
    student_features = student_features.merge(consec_agg, on='UserId', how='left')
else:
    student_features['consecutive_correct_max'] = 0

feature_cols = ['overall_accuracy', 'total_interactions', 'unique_constructs', 
                'unique_subjects', 'accuracy_slope', 'engagement_mean', 'consecutive_correct_max']

X = student_features[feature_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=30)
student_features['student_cluster'] = kmeans.fit_predict(X_scaled)

cluster_sizes = student_features['student_cluster'].value_counts()
small_clusters = cluster_sizes[cluster_sizes < 100].index.tolist()
if small_clusters:
    largest_cluster = cluster_sizes.idxmax()
    student_features.loc[student_features['student_cluster'].isin(small_clusters), 'student_cluster'] = largest_cluster

df = df.merge(
    student_features[['UserId', 'student_cluster']], 
    on='UserId', 
    how='left'
)
clean_main_df = df.reset_index(drop=True)

In [ ]:
# question_difficulty creation
question_stats = df.groupby('QuestionId').agg(
    question_difficulty=('IsCorrect', lambda x: 1 - (x == 1).mean())
).reset_index()

question_stats['question_difficulty'] = question_stats['question_difficulty'].clip(0, 1)

if 'question_difficulty' in df.columns:
    df = df.drop(columns=['question_difficulty'])

df = df.merge(question_stats[['QuestionId', 'question_difficulty']], 
              on='QuestionId', how='left')


scaler = StandardScaler()
df['question_difficulty'] = scaler.fit_transform(df[['question_difficulty']])

clean_main_df = df.reset_index(drop=True)

In [ ]:
# Drop identifiers
drop_cols = [
    'QuizSessionId', 'AnswerId', 'QuestionId',
    'QuizId', 'window_id',
    'AnswerValue', 'CorrectAnswer', 'QuestionSequence',
    'SubjectName', 'QuestionSubjectIds', 'PrerequisiteConstructIds',
    'CheckinQuestionId', 'CheckoutQuestionId'
]

drop_cols = [col for col in drop_cols if col in df.columns]

df = df.drop(columns=drop_cols, errors='ignore')

clean_main_df = df.reset_index(drop=True)

In [ ]:
#Fill null
if pd.api.types.is_datetime64_any_dtype(df['MonthOfBirth']):
    df['MonthOfBirth'] = df['MonthOfBirth'].dt.month
else:
    df['MonthOfBirth'] = pd.to_datetime(df['MonthOfBirth'], errors='coerce').dt.month

df['MonthOfBirth'] = df['MonthOfBirth'].fillna(df['MonthOfBirth'].median())

df['Gender'] = df['Gender'].fillna('missing')
df['YearGroup'] = df['YearGroup'].fillna(-1)
df['Level'] = df['Level'].fillna(-1)

numeric_cols = ['time_delta', 'engagement_proxy', 'consecutive_correct', 'attempts_on_same_question']
for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

df = df.fillna(0)
clean_main_df = df.reset_index(drop=True)

In [ ]:
clean_main_df

# 2. Experimental_df

In [ ]:
#Experiment DF
constructs_test = data["constructs_test"]
construct_experiments_test = data["construct_experiments_test"]
checkin_to_checkout = data["checkin_to_checkout"]

experimental_df = constructs_test.merge(
    construct_experiments_test.drop(columns=['ControlLessonConstructIds'], errors='ignore'),
    how='left',
    on='QuestionConstructId'
)

experimental_df = experimental_df.merge(
    checkin_to_checkout.fillna(0),
    how='left',
    left_on='QuestionConstructId',
    right_on='LessonConstructId'
)

experimental_df = experimental_df.drop(
    columns=['LessonConstructId'],
    errors='ignore'
)

In [ ]:
df = experimental_df.copy()

df = df.dropna(axis='columns', how='all')

def parse_control_lessons(val):
    if pd.isna(val):
        return val
    val_str = str(val).strip('{}')
    return [int(x.strip()) for x in val_str.split(',')] if val_str else []

if 'ControlLessonConstructIds' in df.columns:
    df['ControlLessonConstructIds_parsed'] = df['ControlLessonConstructIds'].apply(parse_control_lessons)

df = df.dropna(subset=['ControlUsersCount', 'TreatmentUsersCount'])

numeric_cols = ['ControlUsersCount', 'TreatmentUsersCount', 
                'TreatmentLessonConstructId_x', 'TreatmentLessonConstructId_y',
                'QuestionConstructId_x', 'QuestionConstructId_y', 'Year_x', 'Year_y']

df[[col for col in numeric_cols if col in df.columns]] = df[[col for col in numeric_cols if col in df.columns]].apply(pd.to_numeric, errors='coerce')

df['TotalUsersCount'] = df['ControlUsersCount'] + df['TreatmentUsersCount']
df['ProportionTreatment'] = df['TreatmentUsersCount'] / df['TotalUsersCount']

if 'ate_p_1' in df.columns and 'ate_k_1' in df.columns:
    df['ATE_difference'] = df['ate_p_1'] - df['ate_k_1']

df = df.reset_index(drop=True)

clean_experimental_df = df

In [ ]:
clean_experimental_df